In [1]:
import sys
sys.path.append('..')

import pandas as pd
import re
from pathlib import Path
from config import engine

def extract_uid(filename):
    """Extract uid from filenames like 'u01.txt' -> 'u01'"""
    match = re.search(r'u\d+', filename)
    return match.group() if match else None

print("✓ Setup complete")

✓ Setup complete


In [2]:
project_root = Path.cwd().parent

# relative path inside the project
folder = project_root / "dataset" / "archive (1)" / "dataset" / "dinning"

print("Project root:", project_root)
print("Dataset folder:", folder)
print("Folder exists:", folder.exists())

list(folder.glob("*.txt"))[:5]

Project root: d:\uni\databases\project
Dataset folder: d:\uni\databases\project\dataset\archive (1)\dataset\dinning
Folder exists: True


[WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u01.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u02.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u04.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u05.txt'),
 WindowsPath('d:/uni/databases/project/dataset/archive (1)/dataset/dinning/u07.txt')]

In [3]:
data = []

for file in folder.glob("*.txt"):
    uid = extract_uid(file.name)

    with open(file, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if line:
            data.append({
                "uid": uid,
                "dinning_event": line
            })

df = pd.DataFrame(data)

df.head()

,uid,dinning_event
0,u01,"2013-01-06 17:42:49,53 Commons,Supper"
1,u01,"2013-01-07 09:32:57,Novack Cafe,Breakfast"
2,u01,"2013-01-07 14:16:07,Courtyard Cafe,Lunch"
3,u01,"2013-01-08 12:51:22,Courtyard Cafe,Lunch"
4,u01,"2013-01-09 13:46:44,King Arthur Flour Coffee B..."


In [4]:
# split dinning_event into datetime, location, meal_type
df[['datetime', 'location', 'meal_type']] = df['dinning_event'].str.split(',', expand=True)

# drop old combined column
df = df.drop(columns=['dinning_event'])

# optional cleanup: strip extra spaces
df['datetime'] = df['datetime'].str.strip()
df['location'] = df['location'].str.strip()
df['meal_type'] = df['meal_type'].str.strip()

# create meal_id
df.insert(0, 'meal_id', range(1, len(df) + 1))

print("✓ Parsed dataframe ready for SQL")
print("Shape after parsing:", df.shape)
print(df.head())


✓ Parsed dataframe ready for SQL
Shape after parsing: (7482, 5)
   meal_id  uid             datetime                      location  meal_type
0        1  u01  2013-01-06 17:42:49                    53 Commons     Supper
1        2  u01  2013-01-07 09:32:57                   Novack Cafe  Breakfast
2        3  u01  2013-01-07 14:16:07                Courtyard Cafe      Lunch
3        4  u01  2013-01-08 12:51:22                Courtyard Cafe      Lunch
4        5  u01  2013-01-09 13:46:44  King Arthur Flour Coffee Bar      Lunch


In [5]:
# insert into SQL table
df.to_sql(
    "dinning",
    engine,
    schema="public",
    if_exists="append",
    index=False
)

print("✓ Data inserted into table 'dinning'")


# verification
check_df = pd.read_sql("SELECT * FROM public.dinning LIMIT 10;", engine)
check_df

✓ Data inserted into table 'dinning'


,meal_id,uid,datetime,location,meal_type
0,1,u01,2013-01-06 17:42:49,53 Commons,Supper
1,2,u01,2013-01-07 09:32:57,Novack Cafe,Breakfast
2,3,u01,2013-01-07 14:16:07,Courtyard Cafe,Lunch
3,4,u01,2013-01-08 12:51:22,Courtyard Cafe,Lunch
4,5,u01,2013-01-09 13:46:44,King Arthur Flour Coffee Bar,Lunch
5,6,u01,2013-01-09 18:33:18,Collis Cafe,Supper
6,7,u01,2013-01-10 17:57:07,53 Commons,Supper
7,8,u01,2013-01-11 10:39:39,King Arthur Flour Coffee Bar,Snack
8,9,u01,2013-01-11 14:01:56,Courtyard Cafe,Lunch
9,10,u01,2013-01-15 11:58:29,Courtyard Cafe,Snack
